In [ ]:
# Parameters (override with papermill)
SEASON = 2025
WEEK   = 10


In [ ]:
import anthropic
import pandas as pd
import json
import os
import re
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

PROJ_PATH = f'fantasy/fantasy_projections/projections_{SEASON}_week{WEEK}.csv'
OUT_PATH  = f'fantasy/agent_analysis_{SEASON}_week{WEEK}.json'

df = pd.read_csv(PROJ_PATH)
print(f'Loaded {len(df)} rows from {PROJ_PATH}')


In [ ]:
def build_table(pos_df):
    rows = []
    for _, r in pos_df.iterrows():
        home = 'vs' if r['is_home'] == 1 else '@'
        if r['injury_status_score'] >= 0.9:
            health = 'Full'
        elif r['injury_status_score'] >= 0.5:
            health = 'Limited'
        else:
            health = 'Questionable'
        row = (
            f"{r['player_display_name']} ({r['team']}) {home} {r['opponent_team']} | "
            f"Proj: {r['projected_pts']:.1f} pts | "
            f"Impl Team Total: {r['implied_team_total']:.1f} | "
            f"Off EPA: {r['off_epa_roll4']:+.3f} (rank #{int(r['off_epa_rank'])}/32) | "
            f"Health: {health}"
        )
        rows.append(row)
    return chr(10).join(rows)

print('build_table ready')


In [ ]:
client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))
results = {}

for pos in ['QB', 'RB', 'WR', 'TE']:
    pos_df = df[df['position'] == pos].copy()
    if pos == 'QB':
        pos_df = pos_df[pos_df['depth_chart_position'] == 1]
        pos_df = pos_df.sort_values('projected_pts', ascending=False).drop_duplicates(subset='team')
    pos_df = pos_df.sort_values('projected_pts', ascending=False).head(20).reset_index(drop=True)

    table = build_table(pos_df)

    prompt_lines = [
        f'You are a sharp fantasy football analyst for a half-PPR league.',
        f'You are writing a SINGLE-WEEK start/sit analysis for Week {WEEK} only.',
        f'Every insight must be about this specific matchup. No season-long value, dynasty, or trade advice.',
        f'Below is Week {WEEK} {pos} data for the top projected players.',
        '',
        'Data key:',
        '- Proj: ML model projected half-PPR points for THIS WEEK',
        '- Impl Team Total: Vegas implied team score for this game (higher = more scoring volume expected)',
        '- Off EPA rank: team 4-game rolling offensive efficiency (1=best offense in NFL, 32=worst)',
        '- Health: Full / Limited / Questionable for this week',
        '',
        'Players (sorted by projected pts):',
        table,
        '',
        'Identify exactly 3 players most likely to OUTPERFORM and exactly 3 most likely to UNDERPERFORM their projection THIS WEEK.',
        'Key signals to reason from:',
        '  - Impl Team Total vs Proj mismatch: a high Vegas team total (26+) with a lower model projection = undervalued.',
        '  - Low team total (20 or under) with a high projection = model may be overrating the player this week.',
        '  - Off EPA rank as a sanity check on team offensive form heading into the game.',
        '  - Health: Limited/Questionable status is a meaningful downside risk.',
        'Keep all reasoning grounded in the data and opponent context. No season narratives.',
        '',
        'Respond ONLY with valid JSON, no markdown fences, no extra text:',
        '{',
        '  "upside": [',
        '    {"player": "Full Name", "team": "XXX", "reason": "1-2 sentence reason."},',
        '    {"player": "Full Name", "team": "XXX", "reason": "1-2 sentence reason."},',
        '    {"player": "Full Name", "team": "XXX", "reason": "1-2 sentence reason."}',
        '  ],',
        '  "downside": [',
        '    {"player": "Full Name", "team": "XXX", "reason": "1-2 sentence reason."},',
        '    {"player": "Full Name", "team": "XXX", "reason": "1-2 sentence reason."},',
        '    {"player": "Full Name", "team": "XXX", "reason": "1-2 sentence reason."}',
        '  ]',
        '}'
    ]
    prompt = chr(10).join(prompt_lines)

    print(f'Calling Claude for {pos}...')
    message = client.messages.create(
        model='claude-opus-4-7',
        max_tokens=1024,
        messages=[{'role': 'user', 'content': prompt}]
    )

    raw = message.content[0].text.strip()
    raw = re.sub(r'^```(?:json)?\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)

    results[pos] = json.loads(raw)
    print(f'  {pos}: {len(results[pos]["upside"])} upside, {len(results[pos]["downside"])} downside')

print('All positions done.')


In [ ]:
results['season'] = SEASON
results['week']   = WEEK
results['generated_at'] = datetime.now().isoformat()

with open(OUT_PATH, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Saved to {OUT_PATH}')
print(json.dumps({k: v for k, v in results.items() if k in ['QB','RB','WR','TE']}, indent=2))
